In [ ]:
%load_ext autoreload
%autoreload 2
from trade.datamanager import (
    DividendDataManager,
    SpotDataManager,
    OptionSpotDataManager,
    VolDataManager,
    RatesDataManager,
    BaseDataManager,
    ForwardDataManager,
    GreekDataManager,
    assert_synchronized_model,
    get_option_theoretical_price,
    calculate_scenarios,
)

from trade.datamanager._enums import (
    OptionSpotEndpointSource,
    SeriesId,
    OptionPricingModel,
    VolatilityModel,
    RealTimeFallbackOption,
    GreekType,
    ModelPrice,
)
from trade.optionlib.config.types import DivType
from trade.helpers.helper_types import SingletonMetaClass
from trade.datamanager.vars import get_loaded_names, TS
from trade.datamanager.utils.model import LoadRequest, _load_model_data_timeseries
from trade.datamanager.utils.logging import (
    change_datamanager_factor_loggers_level,
    change_datamanager_utils_logging_level,
    change_logging_in_all_datamanager_loggers,
    change_all_optionlib_loggers_level,
)

# change_datamanager_factor_loggers_level("CRITICAL")
# change_datamanager_utils_logging_level("CRITICAL")
# change_logging_in_all_datamanager_loggers("CRITICAL")
change_all_optionlib_loggers_level("CRITICAL")
get_loaded_names()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


set()

In [7]:
## Vars
div = DivType.CONTINUOUS
undo_adjust = True
endpoint_source = OptionSpotEndpointSource.EOD
series_id = SeriesId.HIST
market_model = OptionPricingModel.BSM
vol_model = VolatilityModel.MARKET
model_price = ModelPrice.ASK

symbol = "SBUX"
expiration = "2026-09-18"
right = "C"
strike = 100.0
ts_start = "2025-01-01"
ts_end = "2026-01-28"

In [8]:
request = LoadRequest(
    symbol=symbol,
    # start_date=ts_start,
    # end_date=ts_end,
    as_of=ts_end,
    expiration=expiration,
    strike=strike,
    right=right,
    series_id=SeriesId.HIST,
    dividend_type=DivType.DISCRETE,
    endpoint_source=OptionSpotEndpointSource.EOD,
    vol_model=VolatilityModel.MARKET,
    market_model=OptionPricingModel.BINOMIAL,
    model_price=ModelPrice.ASK,
    load_spot=True,
    load_dividend=True,
    load_forward=True,
    load_option_spot=True,
    load_vol=True,
    load_greek=True,
    load_rates=True,
    undo_adjust=True,
    # rt=True,
)
data_packet = _load_model_data_timeseries(request)


2026-02-01 22:13:13 [test] trade.datamanager.vars INFO: Loading timeseries for SBUX...
2026-02-01 22:13:13 [test] EventDriven.riskmanager.market_data INFO: Timeseries for SBUX already loaded. Use force=False to reload.
2026-02-01 22:13:13 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:13:13 [test] trade.datamanager.dividend INFO: Cache hit for key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE
2026-02-01 22:13:13 [test] trade.datamanager.dividend INFO: Cache partially covers requested date range. Key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE. Fetching missing data.
2026-02-01 22:13:15 [test] trade.datamanager.rates INFO: Cache hit for risk-free rate timeseries key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D
2026-02-01 22:13:15 [test] trade.datama

In [9]:
data_packet.greek.fallback_option


<RealTimeFallbackOption.USE_LAST_AVAILABLE: 'use_last_available'>

In [10]:
request.end_date

Timestamp('2026-01-28 00:00:00')

In [12]:
bsm = calculate_scenarios(
    symbol=symbol,
    expiration=expiration,
    right=right,
    strike=strike,
    return_pnl_in_pct=True,
    return_pnl=True,
    rt=True
)


2026-02-01 22:14:01 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:01 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:01 [test] trade.datamanager.dividend WARNING: Valuation date 2026-02-01 22:14:01.847656 is not a business day or holiday. No dividends available. Resolution: RealTimeFallbackOption.USE_LAST_AVAILABLE
2026-02-01 22:14:01 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:01 [test] trade.datamanager.dividend INFO: Cache hit for key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE
2026-02-01 22:14:01 [test] trade.datamanager.dividend INFO: Cache partially covers requested date range. Key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE. Fetching missing data.
2026-02-01 22:14:01 [test] trade.d

In [13]:
bsm.grid

,0.90,0.95,1.00,1.05,1.10
-0.02,-0.594724,-0.376008,-0.093914,0.254399,0.668405
-0.01,-0.559473,-0.334461,-0.046887,0.305528,0.721803
0.00,-0.524432,-0.293099,-0.000001,0.356573,0.775182
0.01,-0.489582,-0.251906,0.046754,0.407539,0.828542
0.02,-0.454905,-0.209543,0.094465,0.458429,0.881879


## Batch Load Real Time

In [14]:
option_metas = [
    {
        "symbol": "SBUX",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 100.0,
    },
    {
        "symbol": "SBUX",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 105.0,
    },
    {
        "symbol": "AAPL",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 260.0,
    },
    {
        "symbol": "AAPL",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 265.0,
    },
    {
        "symbol": "AMD",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 280.0,
    },
    {
        "symbol": "AMD",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 290.0,
    },
    {
        "symbol": "META",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 890.0,
    },
    {
        "symbol": "META",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 900.0,
    }
]

In [15]:
full_data = []
for option_meta in option_metas:
    option_scenarios = calculate_scenarios(
        symbol=option_meta["symbol"],
        expiration=option_meta["expiration"],
        right=option_meta["right"],
        strike=option_meta["strike"],
        return_pnl_in_pct=True,
        return_pnl=True,
        rt=True
    )
    full_data.append(option_scenarios)

2026-02-01 22:14:08 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:08 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:08 [test] trade.datamanager.dividend WARNING: Valuation date 2026-02-01 22:14:08.707741 is not a business day or holiday. No dividends available. Resolution: RealTimeFallbackOption.USE_LAST_AVAILABLE
2026-02-01 22:14:08 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:08 [test] trade.datamanager.dividend INFO: Cache hit for key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE
2026-02-01 22:14:08 [test] trade.datamanager.dividend INFO: Cache partially covers requested date range. Key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE. Fetching missing data.
2026-02-01 22:14:08 [test] trade.d

In [16]:
timeseries_full_data = []
for option_meta in option_metas:
    request = LoadRequest(
        symbol=option_meta["symbol"],
        start_date=ts_start,
        end_date=ts_end,
        expiration=option_meta["expiration"],
        strike=option_meta["strike"],
        right=option_meta["right"],
        series_id=SeriesId.HIST,
        dividend_type=DivType.DISCRETE,
        endpoint_source=OptionSpotEndpointSource.EOD,
        vol_model=VolatilityModel.MARKET,
        market_model=OptionPricingModel.BINOMIAL,
        model_price=ModelPrice.ASK,
        load_spot=True,
        load_dividend=True,
        load_forward=True,
        load_option_spot=True,
        load_vol=True,
        load_greek=True,
        load_rates=True,
        undo_adjust=True,
        # rt=True,
    )
    data_packet = _load_model_data_timeseries(request)
    timeseries_full_data.append(data_packet)


2026-02-01 22:14:24 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:24 [test] trade.datamanager.dividend INFO: Using provided dividend_type: DivType.DISCRETE
2026-02-01 22:14:24 [test] trade.datamanager.dividend INFO: Fetching discrete dividend schedule timeseries for SBUX from 2025-01-01 00:00:00 to 2026-01-28 00:00:00 with maturity 2026-09-18 00:00:00
2026-02-01 22:14:24 [test] trade.datamanager.dividend INFO: Cache hit for discrete schedule timeseries key: symbol:SBUX|interval:eod|artifact_type:divs|series_id:hist|current_state:SCHEDULE_TIMESERIES|lookback_years:1|maturity:2026-09-18|method:CONSTANT|undo_adjust:1
2026-02-01 22:14:24 [test] trade.datamanager.dividend INFO: Cache fully covers requested date range for timeseries. Key: symbol:SBUX|interval:eod|artifact_type:divs|series_id:hist|current_state:SCHEDULE_TIMESERIES|lookback_years:1|maturity:2026-09-18|method:CONSTANT|undo_adjust:1
2026-02-01 22:14:24 [test] trade.datamanager.rates INF

In [17]:
timeseries_full_data


[ModelResultPack(symbol='SBUX', strike=100.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='SBUX', strike=105.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AAPL', strike=260.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AAPL', strike=265.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AMD', strike=280.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId

In [18]:
rt_full_data = []
for option_meta in option_metas:
    request = LoadRequest(
        symbol=option_meta["symbol"],

        expiration=option_meta["expiration"],
        strike=option_meta["strike"],
        right=option_meta["right"],
        series_id=SeriesId.HIST,
        dividend_type=DivType.DISCRETE,
        endpoint_source=OptionSpotEndpointSource.EOD,
        vol_model=VolatilityModel.MARKET,
        market_model=OptionPricingModel.BINOMIAL,
        model_price=ModelPrice.ASK,
        load_spot=True,
        load_dividend=True,
        load_forward=True,
        load_option_spot=True,
        load_vol=True,
        load_greek=True,
        load_rates=True,
        undo_adjust=True,
        rt=True,
    )
    data_packet = _load_model_data_timeseries(request)
    rt_full_data.append(data_packet)


2026-02-01 22:14:33 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:33 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:33 [test] trade.datamanager.dividend WARNING: Valuation date 2026-02-01 22:14:33.336442 is not a business day or holiday. No dividends available. Resolution: RealTimeFallbackOption.USE_LAST_AVAILABLE
2026-02-01 22:14:33 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-01 22:14:33 [test] trade.datamanager.dividend INFO: Cache hit for key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE
2026-02-01 22:14:33 [test] trade.datamanager.dividend INFO: Cache partially covers requested date range. Key: symbol:SBUX|interval:na|artifact_type:divs|series_id:hist|current_state:SCHEDULE|lookback_years:1|method:CONSTANT|vendor:YFINANCE. Fetching missing data.
2026-02-01 22:14:33 [test] trade.d

In [19]:
rt_full_data[0].option_spot.price

datetime
2026-01-30    5.9
Name: midpoint, dtype: float64

In [ ]:
OptionSpotDataManager("SBUX").rt(
    strike=100.0,
    right="C",
    expiration="2026-09-18",
).price

2026-02-01 20:26:45 [test] trade.datamanager.option_spot INFO: Real-time data not available for SBUX on 2026-01-30 00:00:00. Market may be closed.
2026-02-01 20:26:45 [test] trade.datamanager.utils INFO: Using cached date range for 2026-01-30 00:00:00 - 2026-01-30 00:00:00 and option tick SBUX20260918C100
2026-02-01 20:26:45 [test] trade.datamanager.utils INFO: Cache hit for timeseries data structure key: symbol:SBUX|interval:eod|artifact_type:option_spot|series_id:hist|endpoint_source:EOD|expiration:20260918T000000|right:C|strike:100
2026-02-01 20:26:45 [test] trade.datamanager.utils INFO: Sanitizing data from 2026-01-30 00:00:00 to 2026-01-30 00:00:00...
2026-02-01 20:26:45 [test] trade.datamanager.option_spot INFO: Cache hit for option spot timeseries key: symbol:SBUX|interval:eod|artifact_type:option_spot|series_id:hist|endpoint_source:EOD|expiration:20260918T000000|right:C|strike:100


datetime
2026-01-30    5.9
Name: midpoint, dtype: float64

In [ ]:
from dbase.database.db_utils import set_environment_context
set_environment_context("long_bbands")
from algo.positions.loaders.position_vars import get_position_data

In [ ]:
pos = get_position_data(force=True)

2026-02-01 21:31:19 [test] algo.positions.loaders.position_vars INFO: Loading position data for today: 2026-01-30
2026-02-01 21:31:19 [test] algo.positions.loaders.position_vars INFO: Loading position data (force=True, refresh=True, eod_block=False, is_today=True, date=2026-01-30)
2026-02-01 21:31:42 [test] algo.positions.loaders.position_vars INFO: Using cached position data (last loaded at 2026-02-01 21:31:42.646283-05:00)


In [ ]:
pos.strategies["long_bbands"].positions[2].position_data.greeks

OptionGreeks(bs_delta=0.06223978453929746, bs_gamma=0.0003526737479830899, bs_theta=-0.006727226776909845, bs_vega=0.05268265069422684, bs_rho=0.0647675488790389, bs_volga=-0.36754222896313593, binomial_delta=0.06148653673587745, binomial_gamma=0.0003619555499767557, binomial_theta=-0.006531186179476123, binomial_vega=0.051931164934861496, binomial_rho=0.06407247036556019, dollar_bs_delta=np.float64(14.546682518500901), dollar_binomial_delta=np.float64(14.370633440966088))